In [3]:
import os
import hashlib

PARTIAL_CHUNK_SIZE = 1024
FULL_READ_CHUNK = 65536

def walk_files(root_dirs):
    all_files = []
    for root_dir in root_dirs:
        for dirpath, dirnames, filenames in os.walk(root_dir):
            for filename in filenames:
                full_path = os.path.join(dirpath, filename)
                if os.path.isfile(full_path):
                    all_files.append(full_path)
    return all_files
    
def group_by_size(file_paths):
    size_map = {}
    for path in file_paths:
        try:
            size = os.path.getsize(path)
        except OSError:
            continue
        if size not in size_map:
            size_map[size] = []
        size_map[size].append(path)
    result = {}
    for size, paths in size_map.items():
        if len(paths) > 1:
            result[size] = paths
    return result
    
def partial_hash(path, chunk_size=PARTIAL_CHUNK_SIZE, algo="md5"):
    hasher = hashlib.new(algo)
    try:
        with open(path, "rb") as f:
            hasher.update(f.read(chunk_size))
    except OSError:
        return None
    return hasher.hexdigest()
    
def full_hash(path, algo="md5"):
    hasher = hashlib.new(algo)
    try:
        with open(path, "rb") as f:
            while True:
                chunk = f.read(FULL_READ_CHUNK)
                if not chunk:
                    break
                hasher.update(chunk)
    except OSError:
        return None
    return hasher.hexdigest()
    
def find_duplicates(root_dirs, algo="md5"):
    all_files = walk_files(root_dirs)
    size_groups = group_by_size(all_files)
    partial_groups = {}
    for size, paths in size_groups.items():
        for path in paths:
            phash = partial_hash(path, algo=algo)
            if phash is not None:
                key = (size, phash)
                if key not in partial_groups:
                    partial_groups[key] = []
                partial_groups[key].append(path)
    filtered_partial_groups = {}
    for key, paths in partial_groups.items():
        if len(paths) > 1:
            filtered_partial_groups[key] = paths
    full_groups = {}
    for paths in filtered_partial_groups.values():
        for path in paths:
            fhash = full_hash(path, algo=algo)
            if fhash is not None:
                if fhash not in full_groups:
                    full_groups[fhash] = []
                full_groups[fhash].append(path)
    result = {}
    for fhash, paths in full_groups.items():
        if len(paths) > 1:
            result[fhash] = sorted(paths)
    return result
    
def format_size(num_bytes):
    size = float(num_bytes)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if size < 1024:
            return f"{size:.2f} {unit}"
        size /= 1024
    return f"{size:.2f} PB"
    
def print_header(text):
    line = "=" * max(50, len(text) + 4)
    print(f"\n{line}\n  {text}\n{line}")
    
def delete_duplicates(duplicates, dry_run=True):
    deleted_count = 0
    freed_space = 0
    for paths in duplicates.values():
        keeper = paths[0]
        for path in paths[1:]:
            if dry_run:
                print(f"[DRY RUN] Would delete: {path}  (keeping: {keeper})")
            else:
                try:
                    size = os.path.getsize(path)
                    os.remove(path)
                    freed_space += size
                    deleted_count += 1
                    print(f"Deleted: {path}")
                except OSError as e:
                    print(f"Could not delete {path}: {e}")
    if dry_run:
        print("\nThis was a dry run - no files were deleted.")
    else:
        print(f"\nDeleted {deleted_count} files, freed {format_size(freed_space)}.")
        
def prompt_for_folders():
    folders = []
    print("Enter folder paths one at a time. Press Enter on a blank line when done.")
    while True:
        path = input(f"Folder {len(folders) + 1} (or press Enter to finish): ").strip()
        if path == "":
            if not folders:
                print("You need to enter at least one folder.")
                continue
            break
        if not os.path.isdir(path):
            print(f"'{path}' is not a valid folder. Try again.")
            continue
        folders.append(path)
    return folders
    
def display_duplicates_interactive(duplicates):
    print_header(f"Found {len(duplicates)} duplicate group(s)")
    file_lookup = {}
    counter = 1
    total_wasted = 0
    for group_num, paths in enumerate(duplicates.values(), start=1):
        file_size = os.path.getsize(paths[0])
        wasted = file_size * (len(paths) - 1)
        total_wasted += wasted
        print(f"\nGroup {group_num} - {len(paths)} identical files, {format_size(file_size)} each:")
        for path in paths:
            tag = "  <- kept by default" if path == paths[0] else ""
            print(f"   [{counter}] {path}{tag}")
            file_lookup[counter] = path
            counter += 1
    print(f"\nTotal space you could free: {format_size(total_wasted)}")
    return file_lookup
    
def handle_manual_selection(duplicates, file_lookup):
    print("\nEnter the numbers of the files you want to DELETE, separated by commas.")
    print("Example: 2,5,6")
    selection = input("Numbers to delete: ").strip()
    if not selection:
        print("No files selected. Nothing deleted.")
        return
    chosen = []
    try:
        for n in selection.split(","):
            n = n.strip()
            if n:
                number = int(n)
                if number not in chosen:
                    chosen.append(number)
    except ValueError:
        print("That wasn't a valid list of numbers. Nothing deleted.")
        return
    paths_to_delete = []
    invalid = []
    for n in chosen:
        if n in file_lookup:
            paths_to_delete.append(file_lookup[n])
        else:
            invalid.append(n)
    if invalid:
        print(f"These numbers don't exist and were ignored: {sorted(invalid)}")
    if not paths_to_delete:
        print("No valid files selected. Nothing deleted.")
        return
    for paths in duplicates.values():
        all_in_list = True
        for p in paths:
            if p not in paths_to_delete:
                all_in_list = False
        if all_in_list:
            print("Warning: this deletes ALL copies of one file - nothing will remain of it.")
            break
    print("\nFiles that will be deleted:")
    for p in paths_to_delete:
        print(f"   {p}")
    if input(f"\nType 'yes' to permanently delete these {len(paths_to_delete)} file(s): ").strip().lower() != "yes":
        print("Cancelled. No files were deleted.")
        return
    freed = 0
    for path in paths_to_delete:
        try:
            size = os.path.getsize(path)
            os.remove(path)
            freed += size
            print(f"Deleted: {path}")
        except OSError as e:
            print(f"Could not delete {path}: {e}")
    print(f"\nFreed {format_size(freed)}.")
    
def ask_deletion_choice(duplicates, file_lookup):
    print_header("What would you like to do?")
    print("  1) Delete ALL duplicates (keeps one copy per group automatically)")
    print("  2) Choose specific files to delete myself")
    print("  3) Do nothing, just exit")
    while True:
        choice = input("\nEnter 1, 2 or 3: ").strip()
        if choice == "1":
            confirm = input("This permanently deletes files. Type 'yes' to confirm: ").strip().lower()
            if confirm == "yes":
                delete_duplicates(duplicates, dry_run=False)
            else:
                print("Cancelled. No files were deleted.")
            return
        elif choice == "2":
            handle_manual_selection(duplicates, file_lookup)
            return
        elif choice == "3":
            print("No changes made. Exiting.")
            return
        else:
            print("Please enter 1, 2 or 3.")
            
def interactive_session():
    print("DUPLICATE FILE FINDER - Interactive Mode")
    folders = prompt_for_folders()
    print(f"\nScanning {len(folders)} location(s)... this may take a moment.\n")
    duplicates = find_duplicates(folders)
    if not duplicates:
        print_header("No duplicate files found - you're all clear!")
        return
    file_lookup = display_duplicates_interactive(duplicates)
    ask_deletion_choice(duplicates, file_lookup)
    print_header("Done")

interactive_session()


DUPLICATE FILE FINDER - Interactive Mode
Enter folder paths one at a time. Press Enter on a blank line when done.


Folder 1 (or press Enter to finish):  C:\Users\Abhinav Gupta\OneDrive\Desktop\pythontestfolder1
Folder 2 (or press Enter to finish):  C:\Users\Abhinav Gupta\OneDrive\Desktop\pythontestfolder2
Folder 3 (or press Enter to finish):  



Scanning 2 location(s)... this may take a moment.


  Found 1 duplicate group(s)

Group 1 - 2 identical files, 7.37 KB each:
   [1] C:\Users\Abhinav Gupta\OneDrive\Desktop\pythontestfolder1\aaron.txt  <- kept by default
   [2] C:\Users\Abhinav Gupta\OneDrive\Desktop\pythontestfolder2\aaron.txt

Total space you could free: 7.37 KB

  What would you like to do?
  1) Delete ALL duplicates (keeps one copy per group automatically)
  2) Choose specific files to delete myself
  3) Do nothing, just exit



Enter 1, 2 or 3:  1
This permanently deletes files. Type 'yes' to confirm:  yes


Deleted: C:\Users\Abhinav Gupta\OneDrive\Desktop\pythontestfolder2\aaron.txt

Deleted 1 files, freed 7.37 KB.

  Done
